## Obtenemos los modelos únicos de procesadores presentes en el dataset

In [4]:
import pandas as pd
import re

df = pd.read_csv("../datos_crudos/dataset_laptops_mexico.csv")

# 2. Expresión regular para atrapar los modelos crudos
# Busca patrones como "i5-1235u", "ryzen 7 5700u", "m2", "n100"
patron_cpu = r"(i[3579][-_]?\d{4,5}[a-z]*|ryzen\s*[3579]\s*\d{4}[a-z]*|m[1-4]\s*(?:pro|max)?|n\d{3})"

procesadores_encontrados = set() # Usamos 'set' para evitar duplicados matemáticamente

for titulo in df["Nombre"].dropna():
    # Pasar a minúsculas y buscar coincidencias
    matches = re.findall(patron_cpu, str(titulo).lower())
    for match in matches:
        procesadores_encontrados.add(match)

# 3. Guardar los resultados únicos en un txt
lista_unica = sorted(list(procesadores_encontrados))

with open("cpus_unicos.txt", "w") as f:
    for cpu in lista_unica:
        f.write(f"{cpu}\n")

print(f"¡Listo! De 35,000 laptops, solo encontramos {len(lista_unica)} modelos únicos de procesador.")

¡Listo! De 35,000 laptops, solo encontramos 204 modelos únicos de procesador.


## Actualización del catálogo

In [ ]:
import pandas as pd
import re
import json
import os

def actualizar_catalogo_automatico(csv_path="../datos_crudos/dataset_laptops_mexico.csv", json_path="catalogo_cpus.json"):
    # 1. Cargar el catálogo existente si ya hay uno
    if os.path.exists(json_path):
        with open(json_path, 'r', encoding='utf-8') as f:
            catalogo_actual = json.load(f)
    else:
        catalogo_actual = []

    # Crear un set con los patrones que ya tenemos para no duplicar
    patrones_existentes = {item['patron'] for item in catalogo_actual}

    # 2. Extraer los procesadores únicos del CSV actual
    df = pd.read_csv(csv_path)
    # Patrón maestro de búsqueda (atrapa Intel Core, AMD Ryzen, Apple M y Serie N)
    patron_maestro = r"(i[3579][-_]?\d{4,5}[a-z]*|ryzen\s*[3579]\s*\d{4}[a-z]*|m[1-4]\s*(?:pro|max)?|n\d{3})"
    
    procesadores_encontrados = set()
    for titulo in df["Nombre"].dropna():
        matches = re.findall(patron_maestro, str(titulo).lower())
        for match in matches:
            procesadores_encontrados.add(match)

    # 3. Identificar cuáles son estrictamente NUEVOS
    nuevos_cpus = procesadores_encontrados - patrones_existentes

    # 4. Construir los datos para el JSON automáticamente
    nuevos_registros = []
    for cpu in nuevos_cpus:
        # Limpiar espacios extra
        cpu_clean = re.sub(r'\s+', ' ', cpu).strip()
        
        modelo_formateado = cpu_clean.upper()
        generacion = "Desconocida"
        
        # --- Lógica de Inferencia para INTEL CORE ---
        if cpu_clean.startswith('i'):
            # Ejemplo: "i5-12450h" -> Busca el "12"
            match_gen = re.search(r'i[3579]-?(\d{1,2})\d{3}', cpu_clean)
            if match_gen:
                num_gen = match_gen.group(1)
                generacion = f"{num_gen}ª Gen"
            # Normalizar nombre (ej: "i5-12450h" -> "Intel Core i5-12450H")
            partes = cpu_clean.split('-') if '-' in cpu_clean else [cpu_clean[:2], cpu_clean[2:]]
            if len(partes) == 2:
                modelo_formateado = f"Intel Core {partes[0].lower()}-{partes[1].upper()}"

        # --- Lógica de Inferencia para AMD RYZEN ---
        elif cpu_clean.startswith('ryzen'):
            # Ejemplo: "ryzen 7 5700u" -> Busca el "5"
            match_gen = re.search(r'ryzen\s*[3579]\s*(\d)\d{3}', cpu_clean)
            if match_gen:
                num_gen = match_gen.group(1)
                generacion = f"Serie {num_gen}000"
            modelo_formateado = f"AMD {cpu_clean.title()}"

        # --- Lógica de Inferencia para APPLE SILICON ---
        elif cpu_clean.startswith('m'):
            generacion = "Apple Silicon"
            modelo_formateado = f"Apple {cpu_clean.title()}"

        # --- Lógica de Inferencia para INTEL N-SERIES ---
        elif cpu_clean.startswith('n'):
            generacion = "Alder Lake-N"
            modelo_formateado = f"Intel Processor {cpu_clean.upper()}"

        # Ensamblar el nuevo bloque JSON
        nuevos_registros.append({
            "patron": cpu,
            "modelo": modelo_formateado,
            "nucleos": "REVISAR", # <- Salvaguarda contra alucinaciones
            "gen": generacion
        })

    # 5. Guardar y actualizar el archivo JSON
    if nuevos_registros:
        # Ordenar alfabéticamente para mantener orden
        catalogo_actual.extend(nuevos_registros)
        catalogo_actual = sorted(catalogo_actual, key=lambda x: x['patron'])
        
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(catalogo_actual, f, indent=4, ensure_ascii=False)
        
        print(f"✅ Éxito: Se encontraron y agregaron {len(nuevos_registros)} procesadores nuevos al JSON.")
        print("💡 Nota: Abre el archivo 'catalogo_cpus.json', busca los que dicen 'REVISAR' en la clave 'nucleos', ponles el número real y guarda el archivo.")
    else:
        print("✅ Tu catálogo está 100% al día. No se detectaron procesadores nuevos en el último scraping.")

# Ejecutar la función
actualizar_catalogo_automatico()

## Imputación de datos de los núcleos

In [14]:
import re
import pandas as pd
import json

# Cargar el catálogo desde el archivo externo
with open("catalogo_cpus.json", "r", encoding="utf-8") as f:
    CATALOGO_CPU_ESTRICTO = json.load(f)

# Limpia el texto y corrige errores de sintaxis/tipeo comunes en títulos.
def normalizar_y_corregir_typos(texto):
    if not isinstance(texto, str) or texto == "Sin título": 
        return ""

    t = texto.lower()

    # 1. Corrección de faltas de ortografía frecuentes en marcas y modelos
    typos_map = {
        r"\brizen\b": "ryzen",
        r"\brayzen\b": "ryzen",
        r"\brizem\b": "ryzen",
        r"\bcor\s+i": "core i",
        r"\bcorei": "core i",
        r"\bintem\b": "intel",
        r"\bintet\b": "intel",
        r"\baple\b": "apple",
        r"\bm\s+([1-4])\b": r"m\1",  # m 4 -> m4
    }
    for patron, reemplazo in typos_map.items():
        t = re.sub(patron, reemplazo, t)

    # 2. Normalización de separadores en Intel Core (ej: "i 7 1235u", "i71235u" -> "i7-1235u")
    t = re.sub(
        r"\bi\s*([3579])\s*[-_]?\s*(\d{4,5}[a-z]*|n\d{3})\b", r"i\1-\2", t
    )

    # 3. Normalización de Intel N (ej: "n 150", "n-150" -> "n150")
    t = re.sub(r"\bn\s*[-_]?\s*(\d{3})\b", r"n\1", t)

    # 4. Normalización de AMD Ryzen (ej: "ryzen 3 7330 u" -> "ryzen 3 7330u")
    t = re.sub(r"\bryzen\s*([3579])\s*[-_]?\s*(\d{4}[a-z]*)\b", r"ryzen \1 \2", t)

    return t


def identificar_procesador_estricto(titulo_original):
    texto_norm = normalizar_y_corregir_typos(titulo_original)
    if not texto_norm:
        return "No especificado", "No especificado", "No especificado"

    # CAPA A: Coincidencia de Certeza Total (100% verificado contra catálogo)
    for cpu in CATALOGO_CPU_ESTRICTO:
        if re.search(cpu["patron"], texto_norm):
            return cpu["modelo"], cpu["nucleos"], cpu["gen"]

    # CAPA B: Identificación de familia genérica sin SKU (Política: No imputar datos no verificados)
    if re.search(r"core\s*i3", texto_norm):
        return "Intel Core i3", "No especificado", "No especificado"
    if re.search(r"core\s*i5", texto_norm):
        return "Intel Core i5", "No especificado", "No especificado"
    if re.search(r"core\s*i7", texto_norm):
        return "Intel Core i7", "No especificado", "No especificado"
    if re.search(r"ryzen\s*3", texto_norm):
        return "AMD Ryzen 3", "No especificado", "No especificado"
    if re.search(r"ryzen\s*5", texto_norm):
        return "AMD Ryzen 5", "No especificado", "No especificado"

    return "No especificado", "No especificado", "No especificado"


def procesar_dataset_determinista_estricto(
    csv_in="../datos_crudos/dataset_laptops_mexico.csv", csv_out="dataset_laptops_mexico_enriquecidos.csv"
):
    df = pd.read_csv(csv_in)

    # Eliminar registros no válidos
    df = df[df["Nombre"] != "Sin título"].copy()

    modelos, nucleos, generaciones = [], [], []

    for _, row in df.iterrows():
        m, n, g = identificar_procesador_estricto(row["Nombre"])
        modelos.append(m)
        nucleos.append(n)
        generaciones.append(g)

    df["Procesador_Modelo"] = modelos
    df["Numero_Nucleos"] = nucleos
    df["Generacion_CPU"] = generaciones

    df.to_csv(csv_out, index=False)
    print(f"Dataset procesado con política estricta en '{csv_out}'.")
    return df


if __name__ == "__main__":
    procesar_dataset_determinista_estricto()

Dataset procesado con política estricta en 'dataset_laptops_mexico_enriquecidos.csv'.
